In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm


In [ ]:
# Base paths
BASE_DIR = Path.cwd().parent
RUN_NAME = "02ng_1"  # Change as needed for different runs

PSM_FILE = BASE_DIR / "processed_data/psm_clean.csv"
BIOSAUR_FILE = BASE_DIR / f"processed_data/biosaur_features/{RUN_NAME}_features.tsv"
OUT_DIR = BASE_DIR / "analysis/ms1_matching"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PROCESSED = BASE_DIR / "processed_data"
OUT_NOTEBOOK = BASE_DIR / "analysis/processed_data"
OUT_NOTEBOOK.mkdir(parents=True, exist_ok=True)


## PART 1: Load Data


In [ ]:
print("Loading Biosaur features...")
features = pd.read_csv(BIOSAUR_FILE, sep="\t")
print(f"  {len(features)} MS1 features")
print(f"  Columns: {list(features.columns)[:8]}")

print("\nLoading PSM table...")
psm = pd.read_csv(PSM_FILE)
print(f"  {len(psm)} PSMs")


In [ ]:
psm.columns


## PART 2: Match PSM with Biosaur Features


In [ ]:
# Tolerances for matching
ppm_tolerance = 10  # parts per million for mass (m/z)
rt_tolerance = 2.5  # minutes for retention time


In [ ]:
print("\nMatching PSM with Biosaur features...")
print(f"  Tolerance: ±{ppm_tolerance} ppm mass, ±{rt_tolerance} min RT")

matched_rows = []

for idx, psm_row in tqdm(psm.iterrows(), total=len(psm)):
    mz_psm = psm_row.get("Calibrated Observed M/Z")
    psm_charge = psm_row["Charge"]
    psm_rt_min = psm_row["Retention"] / 60.0  # convert seconds to minutes

    # default (no match)
    match = {
        **psm_row.to_dict(),
        "MS1_Intensity": np.nan,
        "MS1_RT": np.nan,
        "BiosaurMZ": np.nan,
        "Biosaur_RT_min": np.nan,
        "Biosaur_intensityApex": np.nan,
        "Biosaur_intensitySum": np.nan,
        "PPMerror": np.nan,
    }

    if pd.isna(mz_psm):
        matched_rows.append(match)
        continue

    feat_charge = features[features["charge"] == psm_charge]

    if len(feat_charge) == 0:
        matched_rows.append(match)
        continue

    # calculate mass difference in ppm using m/z
    ppm_values = (mz_psm - feat_charge["mz"]) / feat_charge["mz"] * 1e6
    feat_charge = feat_charge.assign(ppm_error=ppm_values)

    # filter by mass tolerance
    feat_mass = feat_charge[feat_charge["ppm_error"].abs() <= ppm_tolerance]

    if len(feat_mass) == 0:
        matched_rows.append(match)
        continue

    # calculate RT difference
    rt_diff = (feat_mass["rtApex"] - psm_rt_min).abs()

    # filter by RT tolerance
    feat_rt = feat_mass[rt_diff <= rt_tolerance]

    if len(feat_rt) == 0:
        matched_rows.append(match)
        continue

    # pick best match (closest RT)
    best_idx = rt_diff.loc[feat_rt.index].idxmin()
    best_feature = feat_mass.loc[best_idx]

    # add matched feature info
    match.update({
        "MS1_Intensity": best_feature["intensityApex"],
        "MS1_RT": best_feature["rtApex"],
        "BiosaurMZ": best_feature["mz"],
        "Biosaur_RT_min": best_feature["rtApex"],
        "Biosaur_intensityApex": best_feature["intensityApex"],
        "Biosaur_intensitySum": best_feature["intensitySum"],
        "PPMerror": best_feature["ppm_error"],
    })

    matched_rows.append(match)


In [ ]:
# Create merged dataframe
merged = pd.DataFrame(matched_rows)

n_total = len(merged)
n_matched = merged["MS1_Intensity"].notna().sum()
pct = 100 * n_matched / n_total

print(f"\nResults:")
print(f"  Total PSMs: {n_total}")
print(f"  Matched with MS1: {n_matched} ({pct:.1f}%)")
print(f"  Not matched: {n_total - n_matched} ({100-pct:.1f}%)")


In [ ]:
# Save merged tables
out_processed = OUT_PROCESSED / "psm_with_ms1.csv"
out_analysis = OUT_DIR / "psm_with_ms1.csv"
out_notebook = OUT_NOTEBOOK / "psm_with_biosaurFeatures.csv"

merged.to_csv(out_processed, index=False)
merged.to_csv(out_analysis, index=False)
merged.to_csv(out_notebook, index=False)
print(f"\n[OK] Saved merged table → {out_processed}")
print(f"[OK] Saved copy for analysis → {out_analysis}")
print(f"[OK] Saved copy for notebooks → {out_notebook}")


## PART 3: Diagnostics


In [ ]:
# PPM error distribution
matched_only = merged[merged["MS1_Intensity"].notna()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(matched_only["PPMerror"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("PPM Error")
axes[0].set_ylabel("Count")
axes[0].set_title("Mass Error Distribution")
axes[0].axvline(0, color="red", linestyle="--", alpha=0.5)

# RT offset
rt_offset = matched_only["Biosaur_RT_min"] - (matched_only["Retention"] / 60.0)
axes[1].hist(rt_offset, bins=50, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("RT Offset (min)")
axes[1].set_ylabel("Count")
axes[1].set_title("RT Offset (Biosaur - PSM)")
axes[1].axvline(0, color="red", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(OUT_DIR / "matching_diagnostics.png", dpi=150)
plt.show()


In [ ]:
# MS1 intensity distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.log10(matched_only["MS1_Intensity"] + 1), bins=50, edgecolor="black", alpha=0.7)
ax.set_xlabel("log10(MS1 Intensity)")
ax.set_ylabel("Count")
ax.set_title("MS1 Intensity Distribution (matched PSMs)")
plt.tight_layout()
plt.savefig(OUT_DIR / "ms1_intensity_dist.png", dpi=150)
plt.show()


## PART 4: Summary Stats


In [ ]:
print("=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"Run: {RUN_NAME}")
print(f"Total PSMs: {n_total}")
print(f"Matched with MS1: {n_matched} ({pct:.1f}%)")
print(f"\nMatching parameters:")
print(f"  PPM tolerance: {ppm_tolerance}")
print(f"  RT tolerance: {rt_tolerance} min")
print(f"\nPPM error stats (matched):")
print(f"  Mean: {matched_only['PPMerror'].mean():.2f}")
print(f"  Std: {matched_only['PPMerror'].std():.2f}")
print(f"  Median: {matched_only['PPMerror'].median():.2f}")
print(f"\nMS1 intensity stats (matched):")
print(f"  Mean: {matched_only['MS1_Intensity'].mean():.2e}")
print(f"  Median: {matched_only['MS1_Intensity'].median():.2e}")
print("=" * 50)


## PART 5: Validation - Chimeric Spectra


In [ ]:
print("\n" + "="*60)
print("VALIDATION: Analyzing chimeric spectra with MS1")
print("="*60)

spec_counts = merged.groupby("Spectrum").size()
chimeric = spec_counts[spec_counts >= 2]

print(f"\nTotal spectra: {len(spec_counts)}")
print(f"Chimeric (2+ PSMs): {len(chimeric)}")


In [ ]:
if len(chimeric) == 0:
    print("No chimeric spectra found!")
else:
    chim_dist = chimeric.value_counts().sort_index()
    print("\nChimeric distribution:")
    for n_psm, count in chim_dist.items():
        print(f"  {n_psm} PSMs: {count} spectra")

    example_spec = chimeric.idxmax()
    n_psms = chimeric[example_spec]

    print(f"\n--- Example: {example_spec} ---")
    print(f"Number of PSMs: {n_psms}")

    spec_psm = merged[merged["Spectrum"] == example_spec].copy()
    spec_psm = spec_psm.sort_values("Hyperscore", ascending=False)

    cols_display = ["Peptide", "Charge", "Hyperscore", "MS1_Intensity", "MS1_RT", "Retention", "Intensity"]
    cols_display = [c for c in cols_display if c in spec_psm.columns]

    print("\nPSMs:")
    print(spec_psm[cols_display].to_string(index=False))

    if "Intensity" in spec_psm.columns:
        ms2_intensities = spec_psm["Intensity"].unique()
        if len(ms2_intensities) == 1:
            print(f"\n⚠️  MS2 Intensity: All SAME ({ms2_intensities[0]:.0f})")
        else:
            print(f"\n✓ MS2 Intensity: Different values")

    if "MS1_Intensity" in spec_psm.columns:
        ms1_intensities = spec_psm["MS1_Intensity"].dropna()
        if len(ms1_intensities) > 0:
            ms1_unique = ms1_intensities.nunique()
            if ms1_unique == 1:
                print(f"\n⚠️  MS1 Intensity (Biosaur): All SAME ({ms1_intensities.iloc[0]:.0f})")
            else:
                print(f"\n✓ MS1 Intensity (Biosaur): {ms1_unique} different values")
                print(f"     Range: {ms1_intensities.min():.0f} - {ms1_intensities.max():.0f}")
        else:
            print("\n⚠️  MS1 Intensity: No matches found")

    out_example = OUT_DIR / f"validation_example_{example_spec.replace('/', '_')}.csv"
    spec_psm.to_csv(out_example, index=False)
    print(f"\n[OK] Saved example → {out_example}")


## PART 6: Plots


In [ ]:
# Hyperscore vs MS1 intensity
plot_data = merged.dropna(subset=["Hyperscore", "MS1_Intensity"])

if len(plot_data) > 0:
    plt.figure(figsize=(7, 5))
    plt.scatter(plot_data["Hyperscore"], plot_data["MS1_Intensity"],
                alpha=0.2, s=5, color="steelblue", edgecolors="none")
    plt.yscale("log")
    plt.xlabel("Hyperscore (MS2 quality)")
    plt.ylabel("MS1 Intensity (Biosaur)")
    plt.title("MS2 quality vs MS1 abundance")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "hyperscore_vs_biosaur_ms1.png", dpi=300)
    plt.show()


In [ ]:
# MS1 intensity chimeric vs non-chimeric
if len(chimeric) > 0:
    merged["is_chimeric"] = merged["Spectrum"].isin(chimeric.index)

    chim_data = merged[merged["is_chimeric"]]["MS1_Intensity"].dropna()
    non_chim_data = merged[~merged["is_chimeric"]]["MS1_Intensity"].dropna()

    plt.figure(figsize=(8, 5))
    plt.hist(non_chim_data, bins=50, alpha=0.6, label=f"Non-chimeric (n={len(non_chim_data)})", color="green")
    plt.hist(chim_data, bins=50, alpha=0.6, label=f"Chimeric (n={len(chim_data)})", color="red")
    plt.xscale("log")
    plt.xlabel("MS1 Intensity (Biosaur)")
    plt.ylabel("Count")
    plt.title("MS1 Intensity: Chimeric vs Non-chimeric PSMs")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "biosaur_ms1_chimeric_comparison.png", dpi=300)
    plt.show()


In [ ]:
# RT matching validation
merged["Retention_min"] = merged["Retention"] / 60.0
rt_matched = merged.dropna(subset=["Retention_min", "MS1_RT"])

if len(rt_matched) > 0:
    plt.figure(figsize=(7, 5))
    plt.scatter(rt_matched["Retention_min"], rt_matched["MS1_RT"],
                alpha=0.3, s=3, color="purple")
    plt.plot([0, rt_matched["Retention_min"].max()], [0, rt_matched["Retention_min"].max()],
             'r--', linewidth=1, label="Perfect match")
    plt.xlabel("PSM Retention Time (min)")
    plt.ylabel("Biosaur Feature RT (min)")
    plt.title(f"RT matching validation (±{rt_tolerance} min tolerance)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "rt_matching_validation.png", dpi=300)
    plt.show()


In [ ]:
# Density heatmap
if len(plot_data) > 0:
    plt.figure(figsize=(7, 5))
    sns.histplot(
        data=plot_data, x="Hyperscore", y="MS1_Intensity",
        bins=(50, 50), log_scale=(False, True), cbar=True, cmap="mako"
    )
    plt.title("Density of MS2 quality vs MS1 abundance")
    plt.xlabel("Hyperscore")
    plt.ylabel("MS1 Intensity (log scale)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "hyperscore_vs_ms1_density.png", dpi=300)
    plt.show()


In [ ]:
# KDE comparison
if len(chimeric) > 0 and len(chim_data) > 0 and len(non_chim_data) > 0:
    plt.figure(figsize=(8, 5))
    sns.kdeplot(np.log10(non_chim_data), label="Non-chimeric", fill=True, alpha=0.5, color="green")
    sns.kdeplot(np.log10(chim_data), label="Chimeric", fill=True, alpha=0.5, color="red")
    plt.xlabel("log10(MS1 Intensity)")
    plt.ylabel("Density")
    plt.title("MS1 intensity distribution: Chimeric vs Non-chimeric")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "ms1_intensity_kde_comparison.png", dpi=300)
    plt.show()


In [ ]:
print("\n✅ MS1 matching with Biosaur complete!")
